<div style="background:linear-gradient(135deg,#143840 0%,#2B6264 100%);border-radius:14px;padding:32px 36px;color:#fff;font-family:'DM Sans',Arial,sans-serif;">
  <div style="font-size:11px;letter-spacing:2px;text-transform:uppercase;color:#FF4B31;font-weight:700;margin-bottom:10px;">Solutions Onboarding &middot; IBOR Training</div>
  <div style="font-size:30px;font-weight:700;line-height:1.15;margin-bottom:10px;">IBOR NB00 &mdash; IBOR Instrument Enrichment & Market Data</div>
  <div style="font-size:15px;color:rgba(255,255,255,.82);max-width:640px;line-height:1.55;">Create every instrument (equities, bonds, OTC derivatives, futures, options) and load 18 months of market data, including dual-format FX and MTM quotes for the SimpleInstrument OTC and zero coupon positions.</div>
</div>

<sub>IBOR pack sequence: **NB00** &nbsp;&rarr;&nbsp; NB01 &nbsp;&rarr;&nbsp; NB02 &nbsp;&rarr;&nbsp; NB03 &nbsp;&rarr;&nbsp; NB04 &nbsp;&rarr;&nbsp; NB05 &nbsp;&rarr;&nbsp; NB06 &nbsp;&rarr;&nbsp; NB07 &nbsp;&rarr;&nbsp; NB08</sub>

# NB00: IBOR Instrument Enrichment & Market Data

Creates all instruments (equities, bonds, OTC derivatives, futures, options) and loads 18 months of market data.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lusid-sdk', 'lumipy', '-q'],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

In [ ]:
import os, json
from datetime import datetime, timedelta, date, timezone
import datetime as dt
import pandas as pd
import lusid as lu
import lusid.models as lm
import lumipy as lp

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.options.display.float_format = "{:,.4f}".format

SCOPE = 'ibor-training-v9'
QUOTE_SCOPE = 'ibor-training-v9-quotes'
DATA_DIR = 'data'

def get_factory(app='lusid'):
    secrets_path = "secrets.json"
    with open(secrets_path) as f:
        secrets = json.load(f)
    api_section = secrets.get("api", {})
    pat = api_section.get("accessToken")
    module = __import__(app)
    if pat:
        config_loaders = [module.extensions.ArgsConfigurationLoader(
            api_url=api_section.get("lusidUrl", ""),
            access_token=pat
        )]
    else:
        config_loaders = [module.extensions.SecretsFileConfigurationLoader(secrets_path)]
    factory = module.extensions.SyncApiClientFactory(config_loaders=config_loaders)
    return factory

def get_lumi():
    secrets_path = "secrets.json"
    with open(secrets_path) as f:
        secrets = json.load(f)
    api_section = secrets.get("api", {})
    pat = api_section.get("accessToken")
    if pat:
        lusid_url = api_section.get("lusidUrl", "")
        lumi_url = lusid_url.replace("/api", "/honeycomb")
        lumi = lp.get_client(access_token=pat, api_url=lumi_url)
    else:
        try:
            lumi = lp.get_client(api_secrets_file=secrets_path)
        except ValueError:
            client_id = os.getenv("FBN_CLIENT_ID")
            client_secret = os.getenv("FBN_CLIENT_SECRET")
            lumi = lp.get_client(client_id=client_id, client_secret=client_secret)
    print('Luminesce client initialised')
    return lumi

lusid_factory = get_factory('lusid')
api_instance = lusid_factory.build(lu.ApplicationMetadataApi)
api_response = api_instance.get_lusid_versions()
print('LUSID Environment Initialised')
print('LUSID API Version :', api_response.build_version)
lumi = get_lumi()

instruments_api = lusid_factory.build(lu.InstrumentsApi)
quotes_api = lusid_factory.build(lu.QuotesApi)
properties_api = lusid_factory.build(lu.PropertyDefinitionsApi)
print('APIs initialised')

---
## Property Definitions
Create instrument and transaction properties used across the training.

In [ ]:
# Create property definitions
prop_defs = [
    ("Instrument", "Sector", "string"), ("Instrument", "Country", "string"),
    ("Instrument", "Exchange", "string"), ("Instrument", "MarketCap", "string"),
    ("Transaction", "Strategy", "string"), ("Transaction", "CustodianAccount", "string"),
    ("Transaction", "Broker", "string"),
    ("Transaction", "SettlementDate", "string"), ("Transaction", "ValueDate", "string"),
    ("Transaction", "TradeTime", "string"), ("Transaction", "ExecutionVenue", "string"),
    ("Transaction", "OrderId", "string"), ("Transaction", "AllocationId", "string"),
    ("Transaction", "CounterpartyId", "string"), ("Transaction", "SettlementInstructionType", "string"),
    ("Transaction", "CommissionAmount", "string"), ("Transaction", "CommissionCurrency", "string"),
    ("Transaction", "TaxAmount", "string"), ("Transaction", "AccruedInterest", "string"),
    ("Transaction", "SettlementStatus", "string"), ("Transaction", "ConfirmationStatus", "string"),
    ("Transaction", "TradeSource", "string"), ("Transaction", "PlaceOfTrade", "string"),
    ("Transaction", "PlaceOfClearing", "string"), ("Transaction", "OriginalOrderType", "string"),
    ("Instrument", "Figi", "string"), ("Instrument", "GicsSector", "string"),
    ("Instrument", "GicsIndustryGroup", "string"), ("Instrument", "GicsIndustry", "string"),
    ("Instrument", "MarketCap", "string"), ("Instrument", "Region", "string"),
    ("Instrument", "AssetClass", "string"), ("Instrument", "InstrumentType", "string"),
    ("Instrument", "IssuerCountry", "string"),
    ("Instrument", "PrimaryExchange", "string"), ("Instrument", "ListingStatus", "string"),
    ("Instrument", "ShareClass", "string"), ("Instrument", "VotingRights", "string"),
    ("Instrument", "DividendFrequency", "string"), ("Instrument", "DividendCurrency", "string"),
    ("Instrument", "LotSize", "string"), ("Instrument", "SettlementCycle", "string"),
    ("Instrument", "PriceSource", "string"), ("Instrument", "IndexMembership", "string"),
    ("Instrument", "CFICode", "string"), ("Instrument", "InstrumentClassification", "string"),
    ("Instrument", "CountryOfRisk", "string"), ("Instrument", "CountryOfIncorporation", "string"),
    ("Instrument", "LegalEntityIdentifier", "string"),
    ("Instrument", "ParValueCurrency", "string"), ("Instrument", "TradingCurrency", "string"),
    ("Instrument", "SettlementCurrency", "string"),
    ("Instrument", "ESGRating", "string"), ("Instrument", "CorporateGovernance", "string"),
    ("Instrument", "FloatPercentage", "string"), ("Instrument", "SharesOutstanding", "string"),
    ("Instrument", "AverageDailyVolume", "string"), ("Instrument", "Beta", "string"),
    ("Instrument", "PriceEarningsRatio", "string"), ("Instrument", "DividendYield", "string"),
    ("Instrument", "BookValuePerShare", "string"), ("Instrument", "RevenueGrowthRate", "string"),
    ("Instrument", "DebtToEquityRatio", "string"), ("Instrument", "ReturnOnEquity", "string"),
    ("Portfolio", "Region", "string"), ("Portfolio", "FundType", "string"),
    ("Portfolio", "PortfolioManager", "string"), ("Portfolio", "Benchmark", "string"),
    ("Portfolio", "InvestmentStrategy", "string"),
]

for domain, code_name, data_type in prop_defs:
    try:
        properties_api.create_property_definition(
            create_property_definition_request=lm.CreatePropertyDefinitionRequest(
                domain=domain, scope=SCOPE, code=code_name,
                display_name=code_name, data_type_id=lm.ResourceId(scope="system", code="string"),
                life_time="Perpetual"
            )
        )
        print(f"  Created: {domain}/{SCOPE}/{code_name}")
    except lu.ApiException as e:
        if 'PropertyAlreadyExists' in str(e.body) or 'AlreadyExists' in str(e.body):
            print(f"  Exists: {domain}/{SCOPE}/{code_name}")
        else:
            print(f"  Error: {domain}/{SCOPE}/{code_name}: {str(e.body)[:200]}")

---
## Load Original Equities (10 Core Holdings)

In [ ]:
# Load 10 core equities with ISINs, CUSIPs, SEDOLs
df_eq = pd.read_csv(f"{DATA_DIR}/ibor_equities.csv")
print(f"Loading {len(df_eq)} core equities...")

eq_defs = {}
for _, row in df_eq.iterrows():
    identifiers = {"ClientInternal": lm.InstrumentIdValue(value=row['ClientInternal'])}
    for id_type in ['Isin', 'Cusip', 'Sedol']:
        if pd.notna(row.get(id_type)) and str(row[id_type]).strip():
            identifiers[id_type] = lm.InstrumentIdValue(value=str(row[id_type]).strip())
    
    eq_defs[row['ClientInternal']] = lm.InstrumentDefinition(
        name=row['Name'],
        identifiers=identifiers,
        definition=lm.Equity(instrument_type="Equity", dom_ccy=row['Currency'])
    )

response = instruments_api.upsert_instruments(scope=SCOPE, request_body=eq_defs)
print(f"  Upserted {len(response.values)} equities, {len(response.failed)} failed")

# Set properties
for _, row in df_eq.iterrows():
    props = []
    for prop_code, csv_col in [("Sector", "Sector"), ("Country", "Country"), ("Exchange", "Exchange"), ("MarketCap", "MarketCap")]:
        if pd.notna(row.get(csv_col)) and str(row[csv_col]).strip():
            props.append(lm.ModelProperty(
                key=f"Instrument/{SCOPE}/{prop_code}",
                value=lm.PropertyValue(label_value=str(row[csv_col]).strip())))
    if props:
        try:
            instruments_api.upsert_instruments_properties(
                scope=SCOPE,
                upsert_instrument_property_request=[
                    lm.UpsertInstrumentPropertyRequest(
                        identifier_type="ClientInternal", identifier=row['ClientInternal'],
                        properties=props)])
        except: pass
print("  Properties set")

---
## Load New Equities (109 from iShares Funds)
Equities from S&P 500, AI Innovation, and Blockchain Technology funds.

In [ ]:
# Load 109 new equities from iShares fund data with full identifier set
df_new = pd.read_csv(f"{DATA_DIR}/ibor_equities_new.csv")
print(f"Loading {len(df_new)} new equities with ISINs, CUSIPs, SEDOLs...")

new_defs = {}
for _, row in df_new.iterrows():
    identifiers = {"ClientInternal": lm.InstrumentIdValue(value=row['ClientInternal'])}
    for id_type in ['Isin', 'Cusip', 'Sedol', 'Figi']:
        if pd.notna(row.get(id_type)) and str(row[id_type]).strip():
            identifiers[id_type] = lm.InstrumentIdValue(value=str(row[id_type]).strip())
    
    new_defs[row['ClientInternal']] = lm.InstrumentDefinition(
        name=row['Name'],
        identifiers=identifiers,
        definition=lm.Equity(instrument_type="Equity", dom_ccy=row['Currency'])
    )

response = instruments_api.upsert_instruments(scope=SCOPE, request_body=new_defs)
print(f"  Upserted {len(response.values)}, {len(response.failed)} failed")

# Set enriched properties (Sector, Country, Exchange, MarketCap, GICS)
for _, row in df_new.iterrows():
    props = []
    for prop_code, csv_col in [("Sector", "Sector"), ("Country", "Country"), ("Exchange", "Exchange"),
                                ("MarketCap", "MarketCap")]:
        if pd.notna(row.get(csv_col)) and str(row[csv_col]).strip():
            props.append(lm.ModelProperty(
                key=f"Instrument/{SCOPE}/{prop_code}",
                value=lm.PropertyValue(label_value=str(row[csv_col]).strip())))
    if props:
        try:
            instruments_api.upsert_instruments_properties(
                scope=SCOPE,
                upsert_instrument_property_request=[
                    lm.UpsertInstrumentPropertyRequest(
                        identifier_type="ClientInternal", identifier=row['ClientInternal'],
                        properties=props)])
        except: pass
print("  Properties set (Sector, Country, Exchange, MarketCap)")

---
## Load Bonds (Vanilla + Complex + Global Aggregate)

In [ ]:
# Vanilla bonds (principal=1, FlowConventions for auto coupon generation)
df_vb = pd.read_csv(f"{DATA_DIR}/ibor_bonds_vanilla.csv")
print(f"Loading {len(df_vb)} vanilla bonds...")

vb_defs = {}
for _, row in df_vb.iterrows():
    # Zero coupon bond cannot be a native Bond (payment_frequency 0Y is rejected).
    # It is created separately below as a SimpleInstrument that prices from MTM quotes.
    if str(row.get('PaymentFrequency','')).strip() in ('0Y','0','None','nan') or float(row.get('CouponRate',0))==0:
        continue
    vb_defs[row['ClientInternal']] = lm.InstrumentDefinition(
        name=row['Name'],
        identifiers={"ClientInternal": lm.InstrumentIdValue(value=row['ClientInternal'])},
        definition=lm.Bond(
            instrument_type="Bond",
            start_date=datetime.strptime(row['StartDate'], "%Y-%m-%d").replace(tzinfo=timezone.utc),
            maturity_date=datetime.strptime(row['MaturityDate'], "%Y-%m-%d").replace(tzinfo=timezone.utc),
            dom_ccy=row['Currency'],
            principal=1.0,
            coupon_rate=float(row['CouponRate'])/100.0,
            flow_conventions=lm.FlowConventions(
                currency=row['Currency'],
                payment_frequency=row['PaymentFrequency'],
                day_count_convention=row['DayCountConvention'],
                roll_convention=str(row['RollConvention']),
                settle_days=int(row.get('SettleDays', 1)),
                reset_days=0, payment_calendars=[], reset_calendars=[],
                leap_days_included=True
            )
        )
    )

response = instruments_api.upsert_instruments(scope=SCOPE, request_body=vb_defs)
print(f"  Upserted {len(response.values)} vanilla bonds")

# Zero coupon bond as a SimpleInstrument (native Bond rejects payment_frequency 0Y).
# Prices purely from MTM quotes loaded later in this notebook.
zcb_rows = df_vb[(df_vb['PaymentFrequency'].astype(str).str.strip().isin(['0Y','0','None','nan'])) | (df_vb['CouponRate'].astype(float)==0)]
for _, row in zcb_rows.iterrows():
    zcb_def = lm.InstrumentDefinition(
        name=row['Name'],
        identifiers={"ClientInternal": lm.InstrumentIdValue(value=row['ClientInternal'])},
        definition=lm.SimpleInstrument(
            instrument_type="SimpleInstrument",
            simple_instrument_type="Bond",
            maturity_date=datetime.strptime(row['MaturityDate'], "%Y-%m-%d").replace(tzinfo=timezone.utc),
            dom_ccy=row['Currency'],
            asset_class="Credit",
            instrument_name=row['Name']))
    r = instruments_api.upsert_instruments(scope=SCOPE, request_body={row['ClientInternal']: zcb_def})
    for k, v in r.values.items():
        print(f"  Zero coupon (SimpleInstrument): {v.name} ({v.lusid_instrument_id})")

# Complex bonds
df_cb = pd.read_csv(f"{DATA_DIR}/ibor_bonds_complex.csv")
print(f"Loading {len(df_cb)} complex bonds...")

cb_defs = {}
for _, row in df_cb.iterrows():
    start = datetime.strptime(row['StartDate'], "%Y-%m-%d").replace(tzinfo=timezone.utc)
    mat = datetime.strptime(row['MaturityDate'], "%Y-%m-%d").replace(tzinfo=timezone.utc)
    pf = str(row['PaymentFrequency']) if pd.notna(row['PaymentFrequency']) else "6M"
    
    fc = lm.FlowConventions(
        currency=row['Currency'], payment_frequency=pf,
        day_count_convention=row['DayCountConvention'],
        roll_convention=str(row['RollConvention']),
        settle_days=int(row.get('SettleDays', 2)),
        reset_days=int(row.get('ResetDays', 0)),
        payment_calendars=[], reset_calendars=[], leap_days_included=True)
    
    schedules = [lm.FixedSchedule(
        schedule_type="FixedSchedule", start_date=start, maturity_date=mat,
        flow_conventions=fc, coupon_rate=float(row['CouponRate'])/100.0,
        notional=1.0, payment_currency=row['Currency'])]
    
    cb_defs[row['ClientInternal']] = lm.InstrumentDefinition(
        name=row['Name'],
        identifiers={"ClientInternal": lm.InstrumentIdValue(value=row['ClientInternal'])},
        definition=lm.ComplexBond(instrument_type="ComplexBond", schedules=schedules)
    )

response = instruments_api.upsert_instruments(scope=SCOPE, request_body=cb_defs)
print(f"  Upserted {len(response.values)} complex bonds")

# Global Aggregate bonds (simple instrument type for training)
df_gagg = pd.read_csv(f"{DATA_DIR}/ibor_bonds_gagg.csv")
print(f"Loading {len(df_gagg)} global aggregate bonds...")

gagg_defs = {}
for _, row in df_gagg.iterrows():
    gagg_defs[row['ClientInternal']] = lm.InstrumentDefinition(
        name=row['Name'],
        identifiers={"ClientInternal": lm.InstrumentIdValue(value=row['ClientInternal'])},
        definition=lm.SimpleInstrument(
            instrument_type="SimpleInstrument",
            dom_ccy=row['Currency'],
            simple_instrument_type="Bond",
            asset_class="Credit"
        )
    )

response = instruments_api.upsert_instruments(scope=SCOPE, request_body=gagg_defs)
print(f"  Upserted {len(response.values)} GAGG bonds")

---
## Load OTC Instruments (IRS + Term Deposit)

In [ ]:
# OTC derivatives as SimpleInstruments.
# Native InterestRateSwap and TermDeposit types invoke LUSID's pricing models, which
# override quote-based pricing and produce unrealistic valuations (the native TermDeposit
# multiplied contract_size against the model value, producing a ~$25 trillion line).
# For training we model both as SimpleInstrument (asset_class="Credit") so they price
# purely from the MTM quotes loaded later in this notebook.

irs_def = lm.InstrumentDefinition(
    name="5Y SOFR IRS Pay Fixed 4%",
    identifiers={"ClientInternal": lm.InstrumentIdValue(value="IBOR-IRS-5Y-SOFR")},
    definition=lm.SimpleInstrument(
        instrument_type="SimpleInstrument",
        simple_instrument_type="Other",
        maturity_date=datetime(2029, 6, 1, tzinfo=timezone.utc),
        dom_ccy="USD",
        asset_class="Credit",
        instrument_name="5Y SOFR IRS Pay Fixed 4%"))

td_def = lm.InstrumentDefinition(
    name="6M USD Term Deposit 5.3%",
    identifiers={"ClientInternal": lm.InstrumentIdValue(value="IBOR-TD-6M-USD")},
    definition=lm.SimpleInstrument(
        instrument_type="SimpleInstrument",
        simple_instrument_type="Other",
        maturity_date=datetime(2025, 1, 1, tzinfo=timezone.utc),
        dom_ccy="USD",
        asset_class="Credit",
        instrument_name="6M USD Term Deposit 5.3%"))

response = instruments_api.upsert_instruments(scope=SCOPE, request_body={
    "IRS": irs_def, "TD": td_def})
for k, v in response.values.items():
    print(f"  Created (SimpleInstrument): {v.name} (LUID: {v.lusid_instrument_id})")

---
## Load Futures & Options

In [ ]:
df_fno = pd.read_csv(f"{DATA_DIR}/ibor_futures_options.csv")
print(f"Loading {len(df_fno)} futures and options...")

fno_defs = {}
for _, row in df_fno.iterrows():
    ci = row['ClientInternal']
    inst_type = row.get('InstrumentType', 'Future')
    expiry = row.get('Expiry', '2025-01-01')
    expiry_dt = datetime.strptime(str(expiry), "%Y-%m-%d").replace(tzinfo=timezone.utc)
    start_dt = expiry_dt - timedelta(days=90)
    
    if inst_type == 'Future':
        # Use LUSID Future instrument type
        fno_defs[ci] = lm.InstrumentDefinition(
            name=row['Name'],
            identifiers={"ClientInternal": lm.InstrumentIdValue(value=ci)},
            definition=lm.Future(
                instrument_type="Future",
                start_date=start_dt,
                maturity_date=expiry_dt,
                identifiers={},
                contract_details=lm.FuturesContractDetails(
                    dom_ccy=row['Currency'],
                    contract_code=ci,
                    contract_size=float(row.get('ContractSize', 1)),
                    exchange_code=row.get('Exchange', 'CME'),
                ),
                contracts=1,
                ref_spot_price=0,
                underlying=lm.ExoticInstrument(
                    instrument_type="ExoticInstrument",
                    content=json.dumps({"underlying": row.get("Underlying", ""), "type": "reference"}),
                    instrument_format=lm.InstrumentDefinitionFormat(source_system="Custom", vendor="IBOR", version="1.0")
                )
            )
        )
        print(f"  Future: {ci} ({row['Name']})")
    
    elif inst_type == 'Option':
        # Use LUSID EquityOption instrument type
        # Need the underlying instrument's LUID
        underlying_ci = row.get('Underlying', '').strip()
        option_type = row.get('OptionType', 'Call')
        strike = float(row.get('StrikePrice', 100))
        
        fno_defs[ci] = lm.InstrumentDefinition(
            name=row['Name'],
            identifiers={"ClientInternal": lm.InstrumentIdValue(value=ci)},
            definition=lm.EquityOption(
                instrument_type="EquityOption",
                start_date=start_dt,
                option_maturity_date=expiry_dt,
                option_settlement_date=expiry_dt + timedelta(days=2),
                delivery_type="Cash",
                option_type=option_type,
                strike=strike,
                dom_ccy=row['Currency'],
                underlying_identifier="ClientInternal",
                code=f"IBOR-{underlying_ci}",
                number_of_shares=float(row.get('ContractSize', 100)),
                option_exercise_type="European",
                equity_option_type="Vanilla"
            )
        )
        print(f"  EquityOption: {ci} ({row['Name']}) {option_type} @ {strike}")

response = instruments_api.upsert_instruments(scope=SCOPE, request_body=fno_defs)
print(f"  Upserted {len(response.values)}, {len(response.failed)} failed")
for k, v in response.failed.items():
    print(f"    FAILED: {k}: {v}")

# Set properties on futures and options
for _, row in df_fno.iterrows():
    props = []
    for prop_code_name, csv_col in [("Sector", "Sector"), ("Exchange", "Exchange"),
                                     ("InstrumentType", "InstrumentType")]:
        if pd.notna(row.get(csv_col)) and str(row[csv_col]).strip():
            props.append(lm.ModelProperty(
                key=f"Instrument/{SCOPE}/{prop_code_name}",
                value=lm.PropertyValue(label_value=str(row[csv_col]).strip())))
    # Add derivative-specific properties
    if row.get('Expiry'):
        props.append(lm.ModelProperty(
            key=f"Instrument/{SCOPE}/SettlementCycle",
            value=lm.PropertyValue(label_value="T+1")))
        props.append(lm.ModelProperty(
            key=f"Instrument/{SCOPE}/AssetClass",
            value=lm.PropertyValue(label_value="Derivatives")))
        props.append(lm.ModelProperty(
            key=f"Instrument/{SCOPE}/Region",
            value=lm.PropertyValue(label_value="North America")))
        props.append(lm.ModelProperty(
            key=f"Instrument/{SCOPE}/CountryOfRisk",
            value=lm.PropertyValue(label_value="US")))
        props.append(lm.ModelProperty(
            key=f"Instrument/{SCOPE}/TradingCurrency",
            value=lm.PropertyValue(label_value=row['Currency'])))
        props.append(lm.ModelProperty(
            key=f"Instrument/{SCOPE}/PriceSource",
            value=lm.PropertyValue(label_value="Exchange")))
    if props:
        try:
            instruments_api.upsert_instruments_properties(
                scope=SCOPE,
                upsert_instrument_property_request=[
                    lm.UpsertInstrumentPropertyRequest(
                        identifier_type="ClientInternal", identifier=row['ClientInternal'],
                        properties=props)])
        except: pass
print("  F&O properties set")

In [ ]:
# (Consolidated) Futures are created correctly in the cell above without the convention field.
print("Futures already created above; no repair step needed.")

---
## Load All Market Data

In [ ]:
# Helper to batch upsert quotes from CSV
def load_quotes_from_csv(filepath, id_type="ClientInternal", id_col="ClientInternal"):
    df = pd.read_csv(filepath)
    quotes = {}
    for i, row in df.iterrows():
        inst_id = str(row.get(id_col, row.get('InstrumentId', '')))
        provider = str(row.get('Provider', 'Client'))
        qt = str(row.get('QuoteType', 'Price'))
        field = str(row.get('Field', 'mid'))
        idt = str(row.get('InstrumentIdType', id_type))
        eff = datetime.fromisoformat(str(row['EffectiveAt'])).replace(tzinfo=timezone.utc)
        
        quotes[f"q_{i}"] = lm.UpsertQuoteRequest(
            quote_id=lm.QuoteId(
                quote_series_id=lm.QuoteSeriesId(
                    provider=provider, instrument_id=inst_id,
                    instrument_id_type=idt, quote_type=qt, field=field),
                effective_at=eff.isoformat()),
            metric_value=lm.MetricValue(value=float(row['Value']), unit=str(row['Unit']))
        )
        
        if len(quotes) >= 2000:
            resp = quotes_api.upsert_quotes(scope=QUOTE_SCOPE, request_body=quotes)
            print(f"    Batch: {len(resp.values)} upserted")
            quotes = {}
    
    if quotes:
        resp = quotes_api.upsert_quotes(scope=QUOTE_SCOPE, request_body=quotes)
        print(f"    Final: {len(resp.values)} upserted")
    return len(df)

print("Loading equity prices (core 10)...")
n = load_quotes_from_csv(f"{DATA_DIR}/ibor_market_data_equity.csv")
print(f"  {n} quotes")

print("Loading equity prices (new 109)...")
n = load_quotes_from_csv(f"{DATA_DIR}/ibor_market_data_new_equities.csv")
print(f"  {n} quotes")

print("Loading bond prices...")
n = load_quotes_from_csv(f"{DATA_DIR}/ibor_market_data_bonds.csv")
print(f"  {n} quotes")

print("Loading GAGG bond prices...")
n = load_quotes_from_csv(f"{DATA_DIR}/ibor_market_data_gagg.csv")
print(f"  {n} quotes")

print("Loading futures/options prices...")
n = load_quotes_from_csv(f"{DATA_DIR}/ibor_market_data_futures_options.csv")
print(f"  {n} quotes")

print("Loading FX rates (daily, CurrencyPair type)...")
n = load_quotes_from_csv(f"{DATA_DIR}/ibor_market_data_fx.csv", id_type="CurrencyPair", id_col="InstrumentId")
print(f"  {n} quotes")

print("Loading SOFR fixings (daily, provider=Lusid)...")
n = load_quotes_from_csv(f"{DATA_DIR}/ibor_market_data_sofr.csv", id_type="RIC", id_col="InstrumentId")
print(f"  {n} quotes")

# Full multi-currency FX load (slash + dot formats, Price + Rate) for the iShares
# portfolios. The FX dependency resolves as Fx:TWD.USD (dot) while quotes are commonly
# stored slash-separated, so BOTH formats are loaded, as BOTH Price and Rate quote types.
print("Loading full multi-currency FX (slash+dot, Price+Rate)...")
fx_rates = {
    "TWD/USD": 0.031, "HKD/USD": 0.128, "CNY/USD": 0.138, "JPY/USD": 0.0067,
    "KRW/USD": 0.00075, "EUR/USD": 1.08, "GBP/USD": 1.27, "CHF/USD": 1.13,
    "SEK/USD": 0.095, "DKK/USD": 0.145, "AUD/USD": 0.65, "CAD/USD": 0.74,
    "ILS/USD": 0.27, "INR/USD": 0.012, "SGD/USD": 0.74, "BRL/USD": 0.20,
}

def _build_fx(fmt_sep):
    reqs = {}
    for pair, rate in fx_rates.items():
        ccy1, ccy2 = pair.split("/")
        fid = f"{ccy1}{fmt_sep}{ccy2}"
        for m in range(18):
            d = datetime(2024, 1, 1) + timedelta(days=m*30)
            for qt in ["Price", "Rate"]:
                reqs[f"fx_{ccy1}{ccy2}_{fmt_sep}_{qt}_{m}"] = lm.UpsertQuoteRequest(
                    quote_id=lm.QuoteId(
                        quote_series_id=lm.QuoteSeriesId(
                            provider="Client", instrument_id=fid,
                            instrument_id_type="CurrencyPair", quote_type=qt, field="mid"),
                        effective_at=d.replace(tzinfo=timezone.utc).isoformat()),
                    metric_value=lm.MetricValue(value=rate, unit=ccy2))
    return reqs

# Load slash and dot SEPARATELY, and report success/failure for each so a rejected
# format is visible rather than silently dropped.
for sep, label in [("/", "slash"), (".", "dot")]:
    reqs = _build_fx(sep)
    resp = quotes_api.upsert_quotes(scope=QUOTE_SCOPE, request_body=reqs)
    nok = len(resp.values); nfail = len(resp.failed)
    print(f"  FX {label}: {nok} loaded, {nfail} failed")
    if nfail:
        # show one example failure to diagnose
        k0 = next(iter(resp.failed)); print(f"    e.g. {k0}: {str(resp.failed[k0])[:120]}")
print(f"  Multi-currency FX done ({len(fx_rates)} pairs, both formats)")


---
## Load IRS and Term Deposit MTM Quotes
OTC derivatives need mark to market quotes for valuation since they cannot be priced via simple lookup.

In [ ]:
# IRS MTM quotes (NPV of the swap position)
irs_mtm = {}
mtm_values = [0, 50000, 95000, 135000, 120000, 110000, 85000, 70000,
              55000, 40000, 25000, 15000, 10000, 5000, -5000, -15000, -25000, -30000]
for i in range(18):
    d = datetime(2024, 1, 1) + timedelta(days=i*30)
    if d >= datetime(2024, 6, 1):
        idx = min(i - 5, len(mtm_values) - 1)
        val = mtm_values[max(0, idx)]
    else:
        val = 0
    irs_mtm[f"irs_{i}"] = lm.UpsertQuoteRequest(
        quote_id=lm.QuoteId(
            quote_series_id=lm.QuoteSeriesId(
                provider="Client", instrument_id="IBOR-IRS-5Y-SOFR",
                instrument_id_type="ClientInternal", quote_type="Price", field="mid"),
            effective_at=d.replace(tzinfo=timezone.utc).isoformat()),
        metric_value=lm.MetricValue(value=val, unit="USD"))

resp = quotes_api.upsert_quotes(scope=QUOTE_SCOPE, request_body=irs_mtm)
print(f"IRS MTM: {len(resp.values)} quotes")

# Term Deposit MTM quotes (principal + accrued interest)
td_mtm = {}
for i in range(18):
    d = datetime(2024, 1, 1) + timedelta(days=i*30)
    if d < datetime(2024, 7, 1):
        val = 0
    elif d >= datetime(2025, 1, 1):
        val = 0
    else:
        days_held = (d - datetime(2024, 7, 1)).days
        val = 5000000 + 5000000 * 0.053 * days_held / 360
    td_mtm[f"td_{i}"] = lm.UpsertQuoteRequest(
        quote_id=lm.QuoteId(
            quote_series_id=lm.QuoteSeriesId(
                provider="Client", instrument_id="IBOR-TD-6M-USD",
                instrument_id_type="ClientInternal", quote_type="Price", field="mid"),
            effective_at=d.replace(tzinfo=timezone.utc).isoformat()),
        metric_value=lm.MetricValue(value=val, unit="USD"))

resp = quotes_api.upsert_quotes(scope=QUOTE_SCOPE, request_body=td_mtm)
print(f"TD MTM: {len(resp.values)} quotes")

# Zero Coupon Bond MTM quotes (accretes from ~0.85 toward par by 2028 maturity)
zcb_mtm = {}
for i in range(18):
    d = datetime(2024, 1, 1) + timedelta(days=i*30)
    frac = i / 48.0
    val = 0.85 + 0.15 * frac
    zcb_mtm[f"zcb_{i}"] = lm.UpsertQuoteRequest(
        quote_id=lm.QuoteId(
            quote_series_id=lm.QuoteSeriesId(
                provider="Client", instrument_id="IBOR-ZCB-2028",
                instrument_id_type="ClientInternal", quote_type="Price", field="mid"),
            effective_at=d.replace(tzinfo=timezone.utc).isoformat()),
        metric_value=lm.MetricValue(value=val, unit="USD"))
resp = quotes_api.upsert_quotes(scope=QUOTE_SCOPE, request_body=zcb_mtm)
print(f"ZCB MTM: {len(resp.values)} quotes")

---
## Copp Clark Calendars

In [ ]:
try:
    df_cc = lumi.run("SELECT * FROM CoppClark.FinancialCentres LIMIT 5", quiet=True)
    print(f"Copp Clark calendars available: {len(df_cc)} centres")
except Exception as e:
    print("Copp Clark calendars not licensed on this domain (non-blocking)")
    print("Bond payment dates will not be adjusted for holidays.")

---
## Verification

In [ ]:
# Verify instrument counts
query = f"SELECT Type, COUNT(*) as Cnt FROM Lusid.Instrument WHERE Scope = '{SCOPE}' GROUP BY Type LIMIT 20"
try:
    df = lumi.run(query, quiet=True)
    print("Instruments by type:")
    display(df)
except Exception as e:
    print(f"Verification query error: {e}")

print("\nNB00 Complete. Proceed to NB01: Portfolio Structure.")